In [ ]:
import warnings,subprocess, os, pandas as pd, geopandas as gpd, numpy as np, seaborn as sns, matplotlib.pyplot as plt, folium
from IPython.display import display
from user_inputs import load_selection, update_selection, select_pca_method, id_column
from street_network_processor import StreetNetworkProcessor
selection = load_selection()                  # inputs and choices recorded by run_analysis.py
crime_col, ctrl_vars, buffer_list_mile = selection["crime_col"], selection["ctrl_vars"], selection["buffer_list_mile"]
id_col = id_column(selection)                 # the neighborhood identifier column chosen in stage 1
from common import explore_pca, select_representatives, FigureSaver, report_missing
from final_dataset import assemble_final_dataset, write_final_dataset
%matplotlib inline
figures = FigureSaver("../output/pca_diagnostics").install()   # every figure shown below is also saved as a PNG
np.set_printoptions(suppress=True)
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
pd.set_option("display.precision", 2)
pd.set_option("display.float_format", "{:.2f}".format)
nb = os.path.basename(__file__) if "__file__" in globals() else "pca_analysis.ipynb"
subprocess.run(["jupyter", "trust", nb])

**Preliminary process**


In [ ]:
gdf = gpd.read_file('../data/final_dataset/extracted_measures.shp')
final_list_net = StreetNetworkProcessor.measure_columns(buffer_list_mile)   # same naming rule as stage 1

**Generates visualization plots for diagnostics or distributions**

Every figure displayed in this notebook is also saved to `output/pca_diagnostics/` (numbered, named after its stage and title), and the map below is kept there as HTML.

In [ ]:
popup_vars = [id_col] + final_list_net + ctrl_vars;gdf_vis = gdf.to_crs(4326)
def fmt(v):
    return f"{v:.2f}" if isinstance(v, (float, int)) and pd.notna(v) else v

features = []
for _, r in gdf_vis.iterrows():
    popup_html = "<br>".join([f"{v}: {fmt(r.get(v, 'NA'))}" for v in popup_vars])
    features.append({
        "type": "Feature",
        "geometry": r.geometry.__geo_interface__,
        "properties": {"id": r[id_col], "popup": popup_html}
    })

geojson = {"type": "FeatureCollection", "features": features}

m = folium.Map(
    [gdf_vis.geometry.centroid.y.mean(), gdf_vis.geometry.centroid.x.mean()],
    zoom_start=11, tiles="OpenStreetMap"
)

folium.GeoJson(
    geojson,
    tooltip=folium.GeoJsonTooltip(fields=["id"], aliases=["ID:"]),
    popup=folium.GeoJsonPopup(fields=["popup"], labels=False, max_width=350)
).add_to(m)

m.save("../output/pca_diagnostics/neighborhood_map.html")   # also kept as an HTML file
m

**Exploratory data inspection.**


In [ ]:
print("📊 Descriptive Statistics\n");display(gdf[final_list_net + ctrl_vars].describe().T.round(3))

**Missing values.**

The component analysis cannot run on missing values (the correlation matrix would be undefined), so the notebook stops here if any selected variable has them, listing the variables and the neighborhoods concerned and what to do. Fix them in the neighborhoods layer, rerun stage 1 with `python run_analysis.py --reuse` (seconds, from the checkpoints) and run this notebook again.

In [ ]:
if not report_missing(gdf, final_list_net, ctrl_vars, id_col=id_col).empty:
    raise SystemExit("Stopped before the component analysis: handle the missing values listed above first.")

**Histogram and bar plot for diagnostics or distributions**


In [ ]:
figures.stage = "distribution"
numerical_columns = gdf[final_list_net + ctrl_vars].select_dtypes(include=['number']).columns;print("histogram and bar plots of each variable")
for col in numerical_columns:
    print(col)
    print('Skew :', round(gdf[final_list_net + ctrl_vars][col].skew(), 2))
    plt.figure(figsize = (10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(gdf[col].dropna(), kde=True); plt.xlabel(col)
    plt.subplot(1, 2, 2)
    sns.boxplot(x = gdf[final_list_net + ctrl_vars][col])
    plt.show()

**Boxplot for diagnostics or distributions**


In [ ]:
selected_cols = ["id"] + final_list_net + ctrl_vars; sub = gdf[selected_cols]
numerical_columns = sub.select_dtypes(include='number').columns.tolist()

if "id" in numerical_columns:
    numerical_columns.remove("id")

for col in numerical_columns:
    print(f"Boxplot for {col}")

    plt.figure(figsize=(10, 5))

    # Boxplot
    sns.boxplot(data=sub, x=col)

    # Overlaid raw points
    sns.stripplot(data=sub, x=col, color='black', jitter=True, alpha=1)

    plt.title(f"Boxplot for {col}")
    plt.tight_layout()
    plt.show()

**PCA: choose the number of components interactively**

PCA is used to collapse each block of variables (street-network measures, socio-spatial controls) into a few uncorrelated components, so that the spatial regression models (fitted outside this repository, see the paper) are not affected by multicollinearity. The number of components is a judgement call, so each block is handled in three stages:

1. **Overview** (independent of *n*): correlation heatmap of the variables, scree plot of all eigenvalues with the Kaiser line and cumulative variance, eigenvalue table and rule-of-thumb counts.
2. **Inspect candidate solutions**: type a number *n* and the full report for that solution is displayed: unrotated and rotated loadings (tables and heatmaps), communalities, variance explained, salient loadings per component, cross-loading variables, variables captured by no component and low communalities. Repeat with other values of *n* as often as needed.
3. **Final choice**: a prompt asks for the number of components to retain (Enter keeps the last one inspected). The choice is saved to `user_selection.json` for the record.

Two settings are asked for first. The **extraction method**: the paper describes the procedure as PCA with varimax rotation, while the original computation used `factor_analyzer`'s default (`minres`, minimum-residual common factor analysis); `principal` gives principal components in the strict sense. The **rotation** is set in the same cell (varimax in the paper). The published study retained 4 network and 9 socio-spatial components.

In [ ]:
pca_method   = select_pca_method()   # asks: principal / minres / ml (see note above)
pca_rotation = "varimax"             # rotation of the retained components (paper: varimax)

**Street-network measures: overview, candidate solutions, final choice**

In [ ]:
figures.stage = "network"
n_net, pca_net = explore_pca(gdf, final_list_net, label="street network",
                             rotation=pca_rotation, method=pca_method)

**Street-network measures: one representative measure per component**

Following the paper (Table 2), the network components are not entered in the regression as scores. Guided by the rotated loadings, one original measure is chosen to stand for each component and enters the models after a log(x + 1) transformation; this keeps the coefficients in interpretable units and preserves the buffer scale of each measure. The prompt lists the top-loading measures per component with their cross-loadings; Enter takes the highest loader. The component scores are still carried in the final dataset for a robustness check.

In [ ]:
network_representatives = select_representatives(pca_net, label="street network")

**Socio-spatial controls: overview, candidate solutions, final choice**

In [ ]:
figures.stage = "controls"
n_ctrl, pca_ctrl = explore_pca(gdf, ctrl_vars, label="socio-spatial controls",
                               rotation=pca_rotation, method=pca_method)

# Record the choices next to the variable selection made in run_analysis.py
update_selection(pca_method=pca_method, pca_rotation=pca_rotation,
                 pca_factors_network=n_net, pca_factors_ctrl=n_ctrl,
                 network_representatives=network_representatives)

**Assemble and write the final modeling dataset** (the end product of this repository)

`final_dataset.shp` / `final_dataset.csv` hold one row per neighborhood with everything the spatial models need: identifier and centroid coordinates, the crime count, the area in acres (exposure offset), the representative street-network measures (raw; log(x + 1) is applied at modeling time), the network component scores (robustness check) and the socio-spatial component scores (controls). `final_dataset_dictionary.csv` documents every column.

In [ ]:
final_gdf, dictionary = assemble_final_dataset(gdf, crime_col, network_representatives, pca_net, pca_ctrl, id_col=id_col)
shp, csv, dic = write_final_dataset(final_gdf, dictionary)
print(f"{len(final_gdf)} rows x {final_gdf.shape[1] - 1} columns written to {shp}, {csv} and {dic}")
display(dictionary)